<a href="https://colab.research.google.com/github/varrahan/gemini-coding-agent/blob/main/W2026/Assignments/A3/SYSC4415_W26_A3_v2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Welcome to Assignment 3  
## 80 Mandatory + 20 Bonus marks

## General Instructions:

This Assignment can be done **in a group of two or individually**.

**YOU MUST JOIN A GROUP ON BRIGHTSPACE TO SUBMIT EVEN IF YOU WORK ON YOUR OWN.**

Please state it explicitly at the beginning of the assignment.

You need only one submission if it's group work.

Please print out values when asked using Python's print() function with f-strings where possible.

Submit your **saved notebook with all the outputs** to Brightspace, but ensure it will produce correct outputs upon restarting and click "runtime" → "run all" with clean outputs. Ensure your notebook displays all answers correctly.

```It is highly recommended that you setup your environment locally and use your laptops or desktop computers to work on this assignment rather than Google Colab as you need proper control of your environment.```

## Your Submission MUST contain your signature at the bottom.

### **Objective:** In this assignment, we build a ReAct agent that facilitates **ML operations and model evaluation**.

This assignment is heavily based on [Tutorial 10](https://github.com/emslaboratory/SYSC4415/blob/master/W2026/Tutorials/T10/T10_SimpleAgent.ipynb).

**Submission:** Submit your Notebook with saved outputs as a *.ipynb* file that adopts this naming convention: ***SYSC4415_W26_A3_NameLastname.ipynb*** on *Brightspace*. No other submission (e.g., through email) will be accepted. (Example file name: ```SYSC4415_W26_A3_NameLastname.ipynb``` or ```SYSC4415_W26_A3_Student1_Student2.ipynb```) The notebook MUST contain saved outputs.

**Runtime tips:**
Agentic programming and API calling can be easily done locally and moved to Colab in the final stages, depending on the implementation of your tools and ML tasks you want to run.

In [ ]:
# Name: Varrahan Uthayan
# Student Number: 101229572

# Name:
# Student Number:

In [ ]:
# Libraries to install - leave this code block blank if this does not apply to you
# Please add a brief comment on why you need the library and what it does


# Imports

Some basic libraries you need are imported here. Make sure you include whatever library you need in this entire notebook in the code block below.

If you are using any library that requires installation, please paste the installation command here.
Leave the code block below if you are not installing any libraries.

In [1]:
# Libraries you might need
# General
import os
import zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import time


# For pre-processing
import torch
from torch.utils.data import Dataset, DataLoader, random_split
import torchvision.transforms as transforms
from torchvision.datasets import ImageFolder

# For modeling
import torch.nn as nn
import torch.optim as optim
from tqdm import tqdm
import torchsummary

# For metrics
from sklearn.metrics import  accuracy_score
from sklearn.metrics import  precision_score
from sklearn.metrics import  recall_score
from sklearn.metrics import  f1_score
from sklearn.metrics import  classification_report
from sklearn.metrics import precision_recall_curve
from sklearn.metrics import  roc_auc_score
from sklearn.metrics import confusion_matrix


# Use the SAME Python interpreter that's running this notebook
# to install the package — avoids version mismatch between
# system Python and the notebook kernel
import subprocess, sys
subprocess.check_call([
    sys.executable,       # e.g. /usr/bin/python3.11
    "-m", "pip",
    "install",
    "--upgrade",          # ensure we get the latest version
    "--quiet",            # suppress verbose pip output
    "llm-connector"
])

# llm_connector.cli contains the one-time project scaffolding tool
from llm_connector.cli import init_project

# Creates this folder structure in your working directory:
#   llm-connector/
#     conf/        ← provider config files go here
#     logs/        ← session logs written here automatically
#     .env.template ← copy this to .env and add your API keys
#
# Safe to call multiple times — skips creation if folder already exists
init_project()


# Agent
from llm_connector import chat_completion, cleanup_resources
from dataclasses import dataclass
import re
import json
import time
from typing import Dict, List, Optional


[!] Directory already exists: /content/llm-connector
    Use --force to overwrite, or delete it manually.


# Task 1: Registration and API Activation (5 marks)

For this particular assignment, we will be using OpenRouter or GroqCloud for LLM inference. This task helps you initiate your first LLM call.

Create a free account on [groq.com](https://groq.com/) or/and [openrouter.com](https://openrouter.ai/) and generate an API Key. Both providers maintain free LLM access.

If you ran the previous cell, then at this point you have the following file structure:

```
├── llm-connector
│   ├── .env.template
│   ├── conf
│   │   ├── llm.yaml
│   │   ├── logs.yaml
│   │   ├── override.yaml.template
│   │   └── security.yaml
│   └── logs
│       └── pricing
└── SYSC4415_W26_A3.ipynb
```
You need to rename your .env.template file to ".env" In the template you will see, and you need the first two unless you want to use other providers:

```
OPENROUTER_API_KEY=YOUR-API-KEY
GROQ_API_KEY=YOUR-API-KEY
GOOGLE_API_KEY=YOUR-API-KEY
OPENAI_API_KEY=YOUR-API-KEY
ANTHROPIC_API_KEY=YOUR-API-KEY
```
Add your API key value to the corresponding variables (OPENROUTER_API_KEY or GROQ_API_KEY).

#### ```You must restart notebook Kernel to start using the .env file```

This Assignement uses open-source ```llm-connector``` ```(pip install llm-connector)``` python module that provides a unified interface to prompt LLMs from different providers (OpenRouter, Groq, OpenAI, Anthropic, Google). It was developed by [Igor Bogdanov](mailto:igorbogdanov@cmail.carleton.ca) and is used in his research projects.

#### If you find the ```llm-connector``` module useful, feel free to star it on GitHub: https://github.com/isbogdanov/llm_connector).

**Please report the issues and feel free to use the module in your agentic projects.**


In [6]:
# Q1a (3 marks)
# Select Provider and Model, set hyper-parameters and test message to test that completion works.
# Hint: Use Tutorial 10 and Providers' Documentation
# Explain each parameter and how each value change influences the LLM's output.

# YOUR ANSWER GOES HERE

# PROVIDER is the platform we are hosting our LLM on, in our case, Groq
# MODEL is the actual LLM model we will be using to make our agent
# TEMPERATURE is the randombess of the model's responses, the higher the temperature, the more random and 'creative' the response
# TOP_P tells the model to select a word from a subset that cumulates to the value TOP_P. The larger the value, the larger the subset and more 'creative' the response
# MAX_TOKENS is the limit of how many tokens should be used per response, and essentially tells us the max length of the model response
# messages is just a list of inputs to the model, specifying the role and the content of the input

# In general you can use any model and provider you like if you have access.
# Make sure you understand that some models hide their reasoning.
# Recommended: llama-3.3-70b-versatile - for Groq, and nvidia/nemotron-3-nano-30b-a3b:free - for OpenRouter
# Try not to send to many requests in order to stay within you free tier limit.

PROVIDER = "groq"
MODEL = "llama-3.3-70b-versatile"
TEMPERATURE = 0.7
TOP_P = 0.9
MAX_TOKENS = 256
messages = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": "How much wood could a wood chuck chuck if a wood chuck could chuck wood."}
]


In [7]:
# Q1b (2 marks)
# Prompt the model using the user role about anything different from the tutorial.

# YOUR ANSWER GOES HERE
response, p, c, t, latency = chat_completion(
    messages=messages,
    provider=(PROVIDER, MODEL),
    temperature=TEMPERATURE,
    top_p=TOP_P,
    max_tokens=MAX_TOKENS
)

print(response, p, c, t, latency)

The classic tongue twister.  The answer, of course, is "a woodchuck would chuck as much wood as a woodchuck could chuck if a woodchuck could chuck wood."

But, just for fun, let's do a little math. Woodchucks, also known as groundhogs, are rodents that burrow in the ground. They are known for their ability to move earth as they create their tunnels and dens.

If we assume that a woodchuck can move about 35 cubic feet of soil in a day (which is a rough estimate), and we convert that to wood, we might estimate that a woodchuck could move about 700 pounds of wood per day (assuming the density of wood is about 0.5-0.7 g/cm³).

So, if a woodchuck could chuck wood, it could potentially move about 700 pounds of wood per day. But, of course, this is all just hypothetical, as woodchucks don't actually chuck wood. They're much more interested in eating grasses, fruits, and veggies, and taking long naps in their cozy burrows. 57 234 291 1.0423245319998387


# Task 2: Agent Implementation (5 marks)

This task contains an implementation of the agent from Tutorial 10. The idea of this task is to make sure you understand how basic LLM-Agent works.


In conversational AI, prompting involves three key roles:
1. **the system role** which provides the foundational instructions and constraints as well as sets the agent's behavior and capabilities;
2. **the user role** delivers the actual queries or commands;
3. **the assistant role** generates contextual, step-by-step responses following the system's guidelines.

This structured approach ensures consistent, controlled interactions where the agent maintains its defined behavior while responding to user needs, with each role serving a specific purpose in the conversation flow. If you dont set system role the model will use its defaults.

In [9]:
# Q2a: (5 marks)
# Explain how agent implementation works, providing comments line by line. And answer the question.
# You are not allowed to copy comments from Tutorial 10.
# This paper might be helpful: https://react-lm.github.io/

# Why do we append the whole conversation for each subsequent call
# and what are the potential negative consequences of this?

# We append the whole conversation for each call in order to create a 'context'. This gives the agent a 'memory' of what previously
# was answered and asked in the conversation. This helps build a short term memory for the agent so that it can use what it has learned
# to better answer and maintain the flow of conversation.

# The negative consequence of this is that after a while, there becomes too many messages for the agent to deal with. This can result in
# API payload limits due to the large number of messages, or what is called 'context rot', where the quality of a LLM's response degrades
# as the number of messages added to the state increases

@dataclass
class AgentState: # State shared throughout the agent
    messages: List[Dict[str, str]]
    system_prompt: str

class ML_Agent:  # Agent class

    def __init__(self, system_prompt: str): # Initialize system and state
        self.state = AgentState( # Set the intial state of the agent based on system prompt input
            messages=[{"role": "system", "content": system_prompt}],
            system_prompt=system_prompt,
        )

    def add_message(self, role: str, content: str) -> None:
        self.state.messages.append({"role": role, "content": content}) # Update state by appending provided role and context

    def execute(self) -> str:
        # Get response and metrics based on response to added message
        response, p, c, t, latency = chat_completion(
            self.state.messages,
            provider=(PROVIDER, MODEL),
            temperature=TEMPERATURE,
            top_p=TOP_P,
            max_tokens=MAX_TOKENS
        )

        if response is None: # Check if response is present
            response = "[No response from model]"

        return response

    def __call__(self, message: str) -> str: # Dunder method to allow class to be called like a function
        self.add_message("user", message) # Add message to agent state
        result = self.execute() # Execute agent based on previous message
        self.add_message("assistant", result) # Store result of agent response back in state
        return result

# Task 3: Tools (29 marks)

Tools are specialized functions that enable AI agents to perform specific actions beyond their inherent capabilities, such as retrieving information, performing calculations, or manipulating data. Agents use tools to decompose complex reasoning into observable steps, extend their knowledge beyond training data, maintain state across interactions, and provide transparency in their decision-making process, ultimately allowing them to solve problems they couldn't tackle through reasoning alone.

Essentially, tools are just callback functions invoked by the agent at the appropriate time during the execution loop.

You need to plan your tools for each particular task your agent is expected to solve.
The Model Evaluation Agent we are building should be able to evaluate the model from the model pool on the specific dataset.

Datasets to use: Penguins, Iris, CIFAR-10

You should be able to tell the agent what to do and watch it display the output of the tools' execution, similar to that in Tutorial 10.

User Prompt examples you should be able to give to your agent and expect it to fulfill the task:
- **Evaluate Linear Regression Model on Iris Dataset**
- **Train a logistic regression model on the Iris dataset**
- **Load the Penguins dataset and preprocess it.**
- **Train a decision tree model on the Penguins dataset and evaluate it.**
- **Load the CIFAR-10 dataset and train Mini-ResNet CNN, visualize results**

Classifier Models for Iris and Penguins (use A1 and early tutorials):
  * Logistic Regression (solver='lbfgs')
  * Decision Tree (max_depth=3)
  * KNN (n_neighbors=5)

Any 2 CNN models of your choice for CIFAR-10 dataset (do some research, don't create anything from scratch unless you want to, use the ones provided by libraries and frameworks)

HINT: It is highly recommended that any code from previous assignments and tutorials be reused for tool implementation.

#### IMPORTANT: DON'T FORGET TO IMPORT LIBRARIES REQUIRED FOR ML TASKS!

In [14]:
# Q3a (4 marks): Implement model_memory tool.
# This tool should provide the agent with details about models or datasets
# Example: when asked about Penguin dataset, the agent can use memory to look up
# the source to obtain the dataset.


# YOUR ANSWER GOES HERE
def model_memory(query: str) -> str:
    """Provides details on datasets and model requirements."""
    kb = {
        "iris": "Tabular. Load: load_iris(). Target: 3 species.",
        "penguins": "Tabular. Load: sns.load_dataset('penguins'). Needs dropna() and encoding.",
        "cifar-10": "Images. Load: torchvision.datasets.CIFAR10. Needs transforms.Normalize.",
        "logistic_regression": "Params: solver='lbfgs'.",
        "decision_tree": "Params: max_depth=3.",
        "knn": "Params: n_neighbors=5."
    }
    query_clean = query.lower().strip()
    return next((v for k, v in kb.items() if k in query_clean), "Topic not in memory.")

In [15]:
# Q3b (4 marks): Implement dataset_loader tool.
# loads dataset after obtaining info from memory
from sklearn.datasets import load_iris
import torchvision

# YOUR ANSWER GOES HERE
def dataset_loader(dataset_name: str) -> str:
    """Loads dataset and returns a structural summary."""
    name = dataset_name.lower()
    if 'iris' in name:
        data = load_iris()
        return f"Iris loaded: {data.data.shape[0]} rows, 4 features."
    elif 'penguin' in name:
        df = sns.load_dataset('penguins').dropna()
        return f"Penguins loaded: {len(df)} rows. Columns: {df.columns.tolist()}."
    elif 'cifar' in name:
        torchvision.datasets.CIFAR10(root='./data', train=True, download=True)
        return "CIFAR-10 downloaded/verified. Ready for transforms."
    return "Unknown dataset."

In [ ]:
# Q3c (4 marks): Implement dataset_preprocessing tool.
# preprocesses the dataset to work with the chosen model, and does the splits


# YOUR ANSWER GOES HERE
def dataset_preprocessing(dataset_name: str, data_object=None) -> str:
    """
    Tool: Preprocesses the dataset and performs train/test splits.
    Handles encoding for Penguins, scaling for Iris/Penguins, and
    tensor conversion/normalization for CIFAR-10.

    Arguments:
        dataset_name (str): 'iris', 'penguins', or 'cifar-10'
        data_object (optional): The raw data loaded by dataset_loader.
    """
    dataset_name = dataset_name.lower().strip()

    try:
        # --- TABULAR DATA PREPROCESSING (Iris & Penguins) ---
        if dataset_name in ['iris', 'penguins']:
            if dataset_name == 'iris':
                from sklearn.datasets import load_iris
                data = load_iris()
                X, y = data.data, data.target
            else:
                # Penguins requires specific cleaning and encoding
                import seaborn as sns
                df = sns.load_dataset('penguins').dropna()
                X = df.drop('species', axis=1)
                y = df['species']

                # Convert categorical features (island, sex) to dummy variables
                X = pd.get_dummies(X, drop_first=True)
                # Encode target labels (Adelie, Chinstrap, Gentoo) to 0, 1, 2
                le = LabelEncoder()
                y = le.fit_transform(y)

            # Split into training and testing sets (80/20)
            X_train, X_test, y_train, y_test = train_test_split(
                X, y, test_size=0.2, random_state=42
            )

            # Standardize features (Mean=0, Variance=1)
            scaler = StandardScaler()
            X_train_scaled = scaler.fit_transform(X_train)
            X_test_scaled = scaler.transform(X_test)

            return (f"Preprocessing complete for {dataset_name}.\n"
                    f"Steps taken: Categorical encoding (if applicable), "
                    f"80/20 Split, and StandardScaler normalization.\n"
                    f"Training samples: {len(X_train_scaled)}, Test samples: {len(X_test_scaled)}.")

        elif dataset_name in ['cifar-10', 'cifar10']:
            # Define the pipeline: Convert to Tensor and Normalize around CIFAR-10 mean/std
            transform = transforms.Compose([
                transforms.ToTensor(),
                transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
            ])

            # Note: In a real agent, we might store these in a global 'state'
            return (f"Preprocessing complete for CIFAR-10.\n"
                    f"Pipeline: transforms.ToTensor() -> transforms.Normalize(mean=0.5, std=0.5).\n"
                    f"Data is now ready for a PyTorch DataLoader.")

        else:
            return f"Error: Preprocessing logic not defined for '{dataset_name}'."

    except Exception as e:
        return f"An error occurred during preprocessing: {str(e)}"

In [ ]:
# Q3d (4 marks): Implement train_model tool.
# trains selected model on selected dataset, the agent should not use this tool
# on datasets and models that cannot work together.



# YOUR ANSWER GOES HERE

In [ ]:
# Q3e (3 marks): Implement evaluate_model tool
# evaluates the models and shows the quality metrics (accuracy, precision, and anything else of your choice)


# YOUR ANSWER GOES HERE

In [ ]:
# Q3f (5 marks): Implement visualize_results tool
# provides results of the training/evaluation, open-ended task (2 plots minimum)


# YOUR ANSWER GOES HERE

In [ ]:
# Q3g (5 marks) Now Provide JSON Schema That Helps LLM Understand your tools, make sure it is valid JSON:

TOOL_SCHEMAS = [
    {
        "name":"",
        "description": "",

        "parameters": {
        # See Tutorial 10
        }
    },
    {
        "name": "",
        "description": "",

        "parameters": {
        # See Tutorial 10
        }
    }
]

# Task 4: System Prompt (10 marks)
A system prompt is essential for guiding an agent's behavior by establishing its purpose, capabilities, tone, and workflow patterns. It acts as the "personality and instruction manual" for the agent, defining the format of interactions (like using Thought/Action/Observation steps in our ML agent), available tools, response styles, and domain-specific knowledge—all while remaining invisible to the end user. This hidden layer of instruction ensures the agent consistently follows the intended reasoning process and operational constraints while providing appropriate and helpful responses, effectively serving as the blueprint for the agent's behavior across all interactions.


In [ ]:
# Q4a (10 marks) Build a system prompt to guide the agent.
# Study Tutorial 10.
# Try to find alternative wording to keep the agent in the desired loop,
# don't just copy the prompt from the tutorial.

# Make sure to reflect proper examples of ReAct loop and Tool use.

# Use the following function:

def create_agent():
    # your system prompt goes inside the multiline string

    tools_json = json.dumps(TOOL_SCHEMAS, indent=2)

    system_prompt = """

    """.strip()

    return ML_Agent(system_prompt)


# Task 5: Set the Agent Loop (16 marks)

Now we are building automation of our Thought/Action/Observation sequence in a single ReAct Loop.


In [ ]:
# Q5a1: (4 marks) Explain why we need the following data structure and function and answer the questions.
# Do not copy-paste tutorial comments
# Fill it in with appropriate values:
KNOWN_ACTIONS = {
   # HINT See Tutorial 10.
}
# Question 1: What happens if no values are provided?


# Question 2: Explain why we need this function and why are strategies suggested in this order?

def extract_json_block(text: str) -> dict | None:
    fence_match = re.search(r"```json\s*(\{.*?\})\s*```", text, re.DOTALL)
    if fence_match:
        try:
            return json.loads(fence_match.group(1))
        except json.JSONDecodeError:
            pass
    brace_match = re.search(r"(\{.*\})", text, re.DOTALL)
    if brace_match:
        try:
            return json.loads(brace_match.group(1))
        except json.JSONDecodeError:
            pass
    return None


In [ ]:
# Q5a2: (4 marks)

# Question 1: Explain why we need the following data structure and function.
# Question 2: Are these strategies useful in our case?
# Question 3: Does it interfere with actual tool calls? How or why not?

def sanitize_observation(text: str) -> str:
    text = re.sub(r'```json\s*\{.*?\}\s*```', '[REDACTED]', text, flags=re.DOTALL)
    text = re.sub(r'\{"type"\s*:', '[REDACTED: {"type":', text)

    return text


In [ ]:
# Q5b: (6 marks) Explain how the ReAct loop works line by line.
# You are not allowed to copy/paste comments from tutorial 10
# This paper might be helpful: https://react-lm.github.io/

number_of_steps = 5 # adjust this number for your implementation, to avoid an infinite loop

def react_loop(question: str, max_turns: int = number_of_steps) -> List[Dict[str, str]]:
    agent = create_agent()

    next_prompt = question
    for turn in range(max_turns):
        result = agent(next_prompt)
        print(result)

        thought_match = re.search(r"Thought:\s*(.+)", result)
        thought_text = thought_match.group(1).strip() if thought_match else ""

        parsed = extract_json_block(result)

        if parsed is None:
            print(f"\n>>> Final Answer (no JSON detected): {result}")
            break

        msg_type = parsed.get("type")

        if msg_type == "answer":
            answer = parsed.get("content", result)
            print(f"\n>>> Final Answer: {answer}")
            break

        elif msg_type == "action":
            tool_name  = parsed.get("tool")
            tool_input = parsed.get("input", {})

            print(f"INFO: Detected action: {tool_name} with input: {tool_input}")

            if tool_name not in KNOWN_ACTIONS:
                print(f"WARNING: Unknown tool '{tool_name}' — feeding error back.")
                next_prompt = (f"Observation: Error — unknown tool '{tool_name}'. "
                               f"Available tools are: {', '.join(KNOWN_ACTIONS.keys())}")
                continue

            arg_value = list(tool_input.values())[0] if tool_input else ""

            print(f"\n ---> Executing {tool_name}({arg_value})")
            try:
                observation = KNOWN_ACTIONS[tool_name](arg_value)

            except Exception as e:
                tool_schema = next(
                    (t for t in TOOL_SCHEMAS if t["name"] == tool_name), None
                )
                observation = (
                    f"Error executing {tool_name}: {e}\n"
                    f"Hint — expected schema: {json.dumps(tool_schema, indent=2)}"
                )
                print(f"SELF-HEAL: Feeding error + schema back to agent")

            observation = sanitize_observation(str(observation))

            print(f"Observation: {observation}")

            next_prompt = f"Observation: {observation}"

        else:
            print(f"WARNING: Unrecognized JSON type '{msg_type}'")
            next_prompt = (f"Observation: Error — unrecognized response type '{msg_type}'. "
                           f"Use 'action' or 'answer'.")

    else:
        print(f"\nWARNING: Reached maximum turns ({max_turns}) without a final answer.")

    return agent.state.messages

# This is just for semantic clarity
def query(initial_prompt: str, max_turns: int = 5) -> List[Dict[str, str]]:
    return react_loop(initial_prompt, max_turns=max_turns)

In [ ]:
# Q5c: (2 marks)
# QUESTION 1: How can we check the whole raw history of the agent's interaction with LLM?


# Task 6: Run your agent (15 marks)

Let's see if your agent works

In [ ]:
# Execute any THREE example prompts using your agent. (Each working prompt example will give you 5 marks, 5x3=15)
# DONT FORGET TO SAVE THE OUTPUT

# User Prompt examples you should be able to give to your agent:
# **Evaluate Linear Regression Model on Iris Dataset**
# **Train a logistic regression model on the Iris dataset**
# **Load the Penguins dataset and preprocess it.**
# **Train a decision tree model on the Penguins dataset and evaluate it.**
# **Load the CIFAR-10 dataset and train Mini-ResNet CNN, visualize results**

# Use this template (You can have something similar, it is just an example):

# Example 1: Prompt
print("\nExample 1: Evaluate Linear Regression Model on Iris Dataset")
print("=" * 50)
task = "Evaluate Linear Regression Model on Iris Dataset"
result = query(task)
print("\n" + "=" * 50 + "\n")

# Example 2

# Example 3


# Task 7: Own tool (Bonus 10 marks)
Not valid without completion of all the previous tasks and tool implementations.

In [ ]:
# Build your own additional ML-related tool and provide an example of interaction with your reasoning agent
# using a prompt of your choice that makes the agent use your tool at one of the reasoning steps.


# Task 8: Finalizing (Bonus 10 marks)

In [ ]:
# Q8a: Why do you need to call this function? (5 marks)
cleanup_resources()

# Q8b: (5 marks)
# How to check how many tokens were spent?
# Print the cost table.

# Hint: Use llm-connector documentation (https://github.com/isbogdanov/llm_connector)

Good luck!

## Signature:
Don't forget to insert your name and student number and execute the snippet below.



In [ ]:
!pip install watermark
# Provide your Signature:
%load_ext watermark
%watermark -a 'Your Name, #Student_Number' -nmv --packages numpy,pandas,sklearn,matplotlib,seaborn,graphviz,groq,torch